In [1]:
import pandas as pd
import numpy as np
from datetime import date
from dateutil.relativedelta import relativedelta
import os
from tqdm import tqdm
from transformers import (
  AutoTokenizer,
  AutoModel,
)
from datasets import Dataset
import torch

In [2]:
data_dir = "bda2024_mid_dataset"

files = os.listdir(data_dir)
print("View all datasets:")
for f in files:
    print(f)

View all datasets:
bda2024_202203-202402_討論數據_dcard.csv
bda2024_202203-202402_內容數據_新聞3.csv
bda2024_微股力_籌碼數據-2年.csv
bda2024_202203-202402_內容數據_新聞2.csv
bda2024_微股力_社群PKTD-2年.csv
bda2024_202203-202402_內容數據_新聞1.csv
bda2024_202203-202402_討論數據_ptt.csv
bda2024_微股力_財報數據-2年.csv
bda2024_微股力_個股交易數據-2年.csv
.ipynb_checkpoints
bda2024_202203-202402_討論數據_mobile01-1.csv
bda2024_202203-202402_討論數據_mobile01-2.csv


In [3]:
def load_df(filepath, preview=True):
    print(f"\n----- Loading {filepath}... -----")
    df = pd.read_csv(filepath)
    print(f"Size of dataframe: {df.shape}")
    print(f"Columns: {list(df.columns)}")
    if preview:
        print(df.head())
    return df

In [4]:
folder_path = 'bda2024_mid_dataset/'

news1_df = load_df(folder_path+"bda2024_202203-202402_內容數據_新聞1.csv", preview=False)
news2_df = load_df(folder_path+"bda2024_202203-202402_內容數據_新聞2.csv", preview=False)
news3_df = load_df(folder_path+"bda2024_202203-202402_內容數據_新聞3.csv", preview=False)
disc_ptt_df = load_df(folder_path+"bda2024_202203-202402_討論數據_ptt.csv", preview=False)

noreply_df = pd.concat([news1_df, news2_df, news3_df, disc_ptt_df], ignore_index=True)

disc_dcard_df = load_df(folder_path+"bda2024_202203-202402_討論數據_dcard.csv", preview=False)
disc_dcard_df.rename(columns={'forum': 'p_type'}, inplace=True)    # Repair column name typo in data
disc_m1_df = load_df(folder_path+"bda2024_202203-202402_討論數據_mobile01-1.csv", preview=False)
disc_m2_df = load_df(folder_path+"bda2024_202203-202402_討論數據_mobile01-2.csv", preview=False)

url_pattern_m = r't=(\d+)'
url_pattern_d = r'p/(\d+)'
disc_m1_df['thread_id'] = disc_m1_df['page_url'].str.extract(url_pattern_m)
disc_m2_df['thread_id'] = disc_m2_df['page_url'].str.extract(url_pattern_m)
disc_dcard_df['thread_id'] = disc_dcard_df['page_url'].str.extract(url_pattern_d)

reply_df = pd.concat([disc_dcard_df, disc_m1_df, disc_m2_df], ignore_index=True)

transaction_df = load_df(folder_path+"bda2024_微股力_個股交易數據-2年.csv", preview=False)
report_df = load_df(folder_path+"bda2024_微股力_財報數據-2年.csv", preview=False)
chip_df = load_df(folder_path+"bda2024_微股力_籌碼數據-2年.csv", preview=False)
transaction_df['stock_symbol'] = transaction_df['stock_symbol'].astype(str)
chip_df['stock_symbol'] = chip_df['stock_symbol'].astype(str)    # Repair mixed data types


----- Loading bda2024_mid_dataset/bda2024_202203-202402_內容數據_新聞1.csv... -----
Size of dataframe: (179449, 9)
Columns: ['id', 'p_type', 's_name', 's_area_name', 'post_time', 'title', 'author', 'content', 'page_url']

----- Loading bda2024_mid_dataset/bda2024_202203-202402_內容數據_新聞2.csv... -----
Size of dataframe: (15114, 9)
Columns: ['id', 'p_type', 's_name', 's_area_name', 'post_time', 'title', 'author', 'content', 'page_url']

----- Loading bda2024_mid_dataset/bda2024_202203-202402_內容數據_新聞3.csv... -----
Size of dataframe: (290929, 9)
Columns: ['id', 'p_type', 's_name', 's_area_name', 'post_time', 'title', 'author', 'content', 'page_url']

----- Loading bda2024_mid_dataset/bda2024_202203-202402_討論數據_ptt.csv... -----
Size of dataframe: (50805, 9)
Columns: ['id', 'p_type', 's_name', 's_area_name', 'post_time', 'title', 'author', 'content', 'page_url']

----- Loading bda2024_mid_dataset/bda2024_202203-202402_討論數據_dcard.csv... -----
Size of dataframe: (231320, 10)
Columns: ['id', 'forum', 

/var/folders/h2/1x1_cb595q18rhk70341f0xm0000gn/T/ipykernel_4307/550884886.py:3: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(filepath)


Size of dataframe: (1154225, 8)
Columns: ['stock_name', 'stock_symbol', 'open', 'high', 'low', 'close', 'volume', 'date']

----- Loading bda2024_mid_dataset/bda2024_微股力_財報數據-2年.csv... -----
Size of dataframe: (16482, 16)
Columns: ['stock_name', 'stock_symbol', 'period', 'gross_profit_margin', 'operating_profit_margin', 'net_profit_margin', 'return_on_equity', 'debt_ratio', 'interest_coverage', 'current_ratio', 'quick_ratio', 'accounts_turnover', 'inventory_turnover', 'eps', 'book_value_per_share', 'date']

----- Loading bda2024_mid_dataset/bda2024_微股力_籌碼數據-2年.csv... -----
Size of dataframe: (998031, 9)
Columns: ['stock_name', 'stock_symbol', 'foreign_investor_bought', 'foreign_investor_sold', 'investment_trust_bought', 'investment_trust_sold', 'dealer_bought', 'dealer_sold', 'date']


/var/folders/h2/1x1_cb595q18rhk70341f0xm0000gn/T/ipykernel_4307/550884886.py:3: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(filepath)


## numerical data

In [5]:
#Only use the stocks appear in both df
left = np.setdiff1d(report_df['stock_name'].unique(), chip_df['stock_name'].unique())
right = np.setdiff1d(chip_df['stock_name'].unique(), report_df['stock_name'].unique())
inner = np.intersect1d(chip_df['stock_name'].unique(), report_df['stock_name'].unique())

all = sum([len(left), len(right), len(inner)])
discard = sum([len(left), len(right)])

print(f'Discarded stocks number: {discard}')
print(f'Retained stocks number: {len(inner)}')
print(f'percent of discarded stocks: {discard/all}')

Discarded stocks number: 635
Retained stocks number: 1834
percent of discarded stocks: 0.25718914540299714


In [6]:
# join on stock_symbol where df_chip.date in (df_fin.date - 3 months)
report_df['date'] = pd.to_datetime(report_df['date']).dt.date
chip_df['date'] = pd.to_datetime(chip_df['date']).dt.date
noreply_df['post_time'] = pd.to_datetime(noreply_df['post_time']).dt.date
reply_df['post_time'] = pd.to_datetime(reply_df['post_time']).dt.date

df_fin = report_df.loc[report_df['period']=='Q']

tmp = pd.merge(chip_df, df_fin, on='stock_name', suffixes=['_chip', '_fin'], how='inner')

#26 stocks dropped in after this line due to the mismatch of date
df_finchip = tmp.loc[(tmp['date_fin'] - relativedelta(months=2) <= tmp['date_chip']) & (tmp['date_chip'] < (tmp['date_fin'] + relativedelta(months=1)))]

## text data

### datas with replies

In [7]:
'''
Remove deleted replies(Dcard)
Remove urls
*Whether to remove replies that are not at the date of main?

*Some replies don't have its main, outer join and fillna?
*Replies and main can have different urls -- fixed, unique id is in url
*Same title problem -- fixed by merging on unique id in url.
*About 40% of the rows don't have any stock name mentioned
*Make stock_name_tagger check stock symbols too
'''

# Separate main content and replies
main = reply_df[reply_df['content_type'] == 'main'][['thread_id', 'post_time', 'title', 'content', 'page_url']].copy()
replies = reply_df[reply_df['content_type'] == 'reply'][['thread_id', 'title', 'content', 'page_url']].copy()

# Remove #公告
main = main[~main['title'].str.contains('#公告', regex=False)]

# Define content to be removed
remove_content = ['已經刪除的內容就像 Dcard 一樣，錯過是無法再相見的！', '此篇文章為轉貼文章，請更新至最新版本觀看完整內容。',
                  '這則留言的文字、圖片或影片，因 交換個人資料（電話、電子郵件、通訊軟體 ID、交友軟體 ID 等），目前已進行移除處理。']

# Filter out the removed content
replies = replies[~replies['content'].isin(remove_content)]

# Regex to remove URLs/ new line symbols
regex = r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\\(\\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+|<BR>|\n'

# Remove URLs from 'content' column in replies
main['content'] = main['content'].str.replace(regex, '', regex=True)
replies['content'] = replies['content'].str.replace(regex, '', regex=True)

# Merge main and replies
df_mapped = pd.merge(main, replies, on='thread_id', suffixes=['_main', '_re'], how='outer')

# Group by 'thread_id' and concatenate replies
replies_grouped = df_mapped.groupby('thread_id')['content_re'].agg(lambda x: ' '.join(x.dropna()))

# Join the grouped replies back to the main DataFrame
main = main.join(replies_grouped, on='thread_id').fillna('')
main.rename(columns={'content_re': 'merged_replies'}, inplace=True)
main['all_content'] = main[['title', 'content', 'merged_replies']].agg(' '.join, axis=1)

In [8]:
def process_duplicated(df):
    # Group by 'title' and 'post_time' and find duplicates by these fields
    duplicates = df.duplicated(subset=['title', 'post_time'], keep=False)
    # Work only with duplicated entries
    df_duplicates = df[duplicates]

    # Find the longest 'all_content' within each group of 'title' and 'post_time'
    idx_to_keep = df_duplicates.groupby(['title', 'post_time'])['all_content'].apply(lambda x: x.idxmax())

    # Filter out duplicates except for the longest 'all_content' entries
    df_clean = df.drop(df_duplicates.index)  # Drop all duplicates
    df_clean = pd.concat([df_clean, df.loc[idx_to_keep]])  # Concatenate entries to keep with non-duplicate data

    return df_clean

In [9]:
# Basic tagger
stock_list = df_finchip['stock_name'].unique()
def stock_name_tagger_1to1(row, stock_list=stock_list):
    title = row['title']
    all_content = row['all_content']

    title_stock = [stock for stock in stock_list if stock in title]
    if len(title_stock) == 1:
        return title_stock[0]

    stock_counts = {}
    for stock in stock_list:
        count = all_content.count(stock)
        if count > 0:
            stock_counts[stock] = count
    if stock_counts:
        return max(stock_counts, key=stock_counts.get)
    return np.nan

In [10]:
# Tagger with Aho–Corasick algorithm(faster)
import ahocorasick

def build_automaton(stock_list):
    # Initialize the automaton
    A = ahocorasick.Automaton()
    for idx, stock in enumerate(stock_list):
        A.add_word(stock, (idx, stock))
    A.make_automaton()
    return A

# Pre-build the automaton with the unique stock names
stock_list = df_finchip['stock_name'].unique()
automaton = build_automaton(stock_list)

def stock_name_tagger_1to1_ac(row, automaton):
    title = row['title']
    all_content = row['all_content']

    # Search for stock names in the title
    title_stocks = set()
    for end_index, (_, stock) in automaton.iter(title):
        title_stocks.add(stock)
    if len(title_stocks) == 1:
        return next(iter(title_stocks))

    # Count occurrences in all_content using the automaton
    content_stock_count = {}
    for end_index, (_, stock) in automaton.iter(all_content):
        if stock in content_stock_count:
            content_stock_count[stock] += 1
        else:
            content_stock_count[stock] = 1

    # Determine the stock name with the maximum count
    if content_stock_count:
        return max(content_stock_count, key=content_stock_count.get)

    return np.nan

In [11]:
tqdm.pandas()
df_reply_merged = process_duplicated(main).drop(columns = 'page_url')
df_reply_1to1 = df_reply_merged.copy()
df_reply_1to1['stock_name'] = df_reply_1to1.progress_apply(stock_name_tagger_1to1, axis=1)
df_reply_1to1['stock_name_ac'] = df_reply_1to1.progress_apply(lambda row: stock_name_tagger_1to1_ac(row, automaton), axis=1)

100%|███████████████████████████████████████████████████████████████| 32234/32234 [00:03<00:00, 8505.15it/s]


In [12]:
mentioned_num = df_reply_1to1[~df_reply_1to1['stock_name'].isnull()].shape[0]
unmentioned_num = df_reply_1to1[df_reply_1to1['stock_name'].isnull()].shape[0]
print(f'Rows with stock name: {mentioned_num}\nRows without stock name: {unmentioned_num}\n{mentioned_num/(unmentioned_num+mentioned_num)}% retained')

used_stocks = len(df_reply_1to1['stock_name'].dropna().unique())
used_stocks_2 = len(df_reply_1to1['stock_name_ac'].dropna().unique())
print(f'{used_stocks/len(stock_list)}% of stock names tagged')
print(f'{used_stocks_2/len(stock_list)}% of stock names tagged')

Rows with stock name: 19859
Rows without stock name: 12375
0.6160886020971645% retained
0.6178906681391496% of stock names tagged
0.6344561016013253% of stock names tagged


### datas without replies

In [13]:
#Remove [公告]
df = noreply_df[~noreply_df['title'].str.contains('[公告]', regex=False)][['id', 'post_time', 'title', 'content']].copy()
df['content'] = df['content'].str.replace(regex, '', regex=True).fillna('')
df['all_content'] = df[['title', 'content']].agg(''.join, axis=1)

df_noreply = process_duplicated(df)
df_noreply['stock_name_ac'] = df_noreply.progress_apply(lambda row: stock_name_tagger_1to1_ac(row, automaton), axis=1)
df_noreply['stock_name'] = df_noreply.progress_apply(stock_name_tagger_1to1, axis=1)

100%|█████████████████████████████████████████████████████████████| 530699/530699 [06:28<00:00, 1365.18it/s]


In [14]:
all_text_data = pd.concat([df_reply_1to1.dropna()[['post_time', 'all_content', 'stock_name']], df_noreply.dropna()[['post_time', 'all_content', 'stock_name']]])

In [15]:
model_name = 'hfl/chinese-bert-wwm-ext'
device = 'mps'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)
max_length = 512


# Function to split the content into max_length chunks
def split_into_chunks(text, max_length):
    return [text[i:i+max_length] for i in range(0, len(text), max_length)]


# Function to get the next trade date
def get_next_date(input_date, dates_list):
    if input_date > dates_list[-1]:
        return np.nan
    index = sum(1 for d in dates_list if d < input_date)
    dates_list[index]
    return dates_list[index]


# Create a new DataFrame to store the chunked data with preserved post_time and stock_name
chunked_data = []
for _, row in all_text_data.iterrows():
    chunks = split_into_chunks(row['all_content'], max_length)
    for chunk in chunks:
        chunked_data.append({
            'post_time': row['post_time'],
            'stock_name': row['stock_name'],
            'content': chunk
        })

transaction_df['date'] = pd.to_datetime(transaction_df['date']).dt.date
dates_list = sorted(transaction_df['date'].unique())

# Merge on trade date so no data is dropped
chunked_df = pd.DataFrame(chunked_data)
chunked_df['trade_date'] = chunked_df['post_time'].progress_apply(lambda x: get_next_date(x, dates_list))
final = pd.merge(chunked_df, df_finchip, left_on=['stock_name', 'trade_date'], right_on=['stock_name', 'date_chip'])
final = final.drop(['date_chip', 'date_fin'], axis=1)

# Convert the DataFrame to a Hugging Face Dataset
hf_dataset = Dataset.from_pandas(final[['post_time', 'stock_name', 'content']])


# Function to tokenize each chunk
def tokenize_function(examples):
    return tokenizer(examples['content'], truncation=True, padding='max_length', max_length=max_length)

# Tokenize the dataset
tokenized_dataset = hf_dataset.map(tokenize_function, batched=True)

100%|███████████████████████████████████████████████████████████| 815063/815063 [00:07<00:00, 108696.96it/s]


Map:   0%|          | 0/655055 [00:00<?, ? examples/s]

In [16]:
def extract_embeddings(batch):
    # Convert to tensors
    inputs = {k: torch.tensor(v) for k, v in batch.items() if k in tokenizer.model_input_names}

    # Move inputs to the device model is on. Remove this if not using CUDA
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # Forward pass, get hidden states
    with torch.no_grad():
        outputs = model(**inputs)

    # Extract the embeddings for the [CLS] token (used as the aggregate sequence representation for classification tasks)
    embeddings = outputs.last_hidden_state[:, 0, :].detach().cpu().numpy()
    return {"embeddings": embeddings}


# Map the dataset to extract embeddings, adjust batch_size for your machine's capabilities
batch_size = 64# Adjust batch size according to your GPU/CPU/RAM
device = 'mps'
model.to(device)
embeddings_dataset = tokenized_dataset.map(extract_embeddings, batched=True, batch_size=batch_size)

Map:   0%|          | 0/655055 [00:00<?, ? examples/s]

In [17]:
np.savetxt('embeddings.csv', embeddings_dataset['embeddings'], fmt='%s', delimiter=",")

In [18]:
final = final.drop(['content'], axis=1).to_numpy()
final = np.hstack((final, embeddings_dataset['embeddings']))
final_df = pd.DataFrame(final)
final_df.to_csv('merged_embeddings.csv', index=False)

In [20]:
x = pd.read_csv('merged_embeddings.csv')
x

,0,1,2,3,4,5,6,7,8,9,...,782,783,784,785,786,787,788,789,790,791
0,2022-03-01,台積電,2022-03-01,2330,33868510,59393909,733000,92000,2206457,827796,...,0.766458,-0.301630,0.206771,-0.236764,0.039337,-0.591708,-0.465070,0.424499,-0.251579,-0.231501
1,2022-03-01,台積電,2022-03-01,2330,33868510,59393909,733000,92000,2206457,827796,...,0.748023,0.200840,-0.105250,-0.214787,-0.147682,-0.956461,-0.167647,0.099332,-0.408404,-0.208983
2,2022-03-01,台積電,2022-03-01,2330,33868510,59393909,733000,92000,2206457,827796,...,0.520980,-0.231023,0.131039,-0.280313,0.490919,-0.824481,-0.045931,0.198517,-0.364945,-0.279433
3,2022-03-01,台積電,2022-03-01,2330,33868510,59393909,733000,92000,2206457,827796,...,0.029969,-0.684525,-0.451997,-0.375910,1.236042,-0.861707,0.227966,0.335897,-0.155650,-0.476788
4,2022-03-01,大量,2022-03-01,3167,43000,40000,0,0,1000,0,...,0.595291,-0.157128,0.162307,-0.345452,0.560773,-0.876018,0.007811,0.240542,0.022666,-0.013727
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
655050,2023-02-09,金利,2023-02-09,5383,0,0,0,0,0,0,...,0.147543,0.112554,0.091180,-0.510792,0.463984,-0.609169,-0.126569,-0.013638,-0.340798,-0.089075
655051,2023-09-27,精確,2023-09-27,3162,57000,124000,0,0,1000,126,...,0.569829,0.496010,0.306804,-0.293615,0.556092,-0.459440,-0.077486,0.075124,-0.521609,0.132624
655052,2023-09-27,精確,2023-09-27,3162,57000,124000,0,0,1000,126,...,0.529539,-0.061102,0.244756,-0.367060,0.537474,-0.608479,-0.390882,-0.026355,-0.581091,0.049368
655053,2023-09-27,精確,2023-09-27,3162,57000,124000,0,0,1000,126,...,0.849503,-0.624550,-0.433148,-0.455792,-0.257379,-0.778337,-0.024688,0.441708,-0.011780,-0.532358
